# 32 - Reading the trade-off curve: why gain 50 wins, and how to use it

**Purpose.** Explain the curve notebook `31` drew for NGC 7000 on 2026-09-05, for someone choosing
a setting at the mount. `31` is written for someone *checking* it; this one is slow on purpose.

It explains three things: how to read the curve at a chosen share of clipped stars; why dead time,
not read noise, decides the gain; and the limits the curve was drawn under.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `pareto_curve.csv` and `pareto_constants.json`, with the model constants `31` used. If
anything here disagrees with `results/`, `results/` is right and this notebook is the bug. Section 3
splits two published points into the model's own factors to show *why* one beats the other; it
does not compute a new curve, and where the gain answer flips with dead time is left for its own
notebook.

**It assumes** `24_model_constants_read.ipynb` for the model, and `28_star_mask_read.ipynb` and
`30_star_clipping_read.ipynb` for the two halves that passed on this night.

**The headline, recomputed in section 1:** gain 50 is the best setting at every share of clipped
stars from 0.25% to about 9%.

In [ ]:
import json
import math
import pathlib
import sys

import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
curve = pd.read_csv(RESULTS / "pareto_curve.csv")
pc = M.Constants.load(RESULTS / "pareto_constants.json")
front = pd.DataFrame(pc.entry("pareto_front")["value"])
mc = json.loads((RESULTS / "model_constants.json").read_text())
T_DEAD = mc["t_dead"]["value"]
print(pc.entry("pareto_front")["note"])

## 1. Reading the curve

Each point on the curve is a setting - a gain and a sub length - that no other setting beats on
both counts: more faint-signal SNR, fewer clipped stars. SNR is a share of the best this night
could give, in blue, the dimmest plane. Clipping is the share of 8033 stars whose core clips in any
plane.

To use it: decide how many stars you are willing to lose, find that share, and read off the
setting.

In [ ]:
rows = []
for budget in (0.0025, 0.005, 0.01, 0.02, 0.05, 0.10):
    ok = front[front.clipped_any <= budget]
    r = ok.loc[ok.snr_rel.idxmax()]
    rows.append({"stars allowed to clip, %": 100 * budget, "gain": int(r.gain), "t, s": r.t,
                 "subs": int(r.n_subs), "SNR, share of best": r.snr_rel,
                 "actually clipped, %": 100 * r.clipped_any})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"\nby gain on the curve: {front.gain.value_counts().sort_index().to_dict()}")
first_high = front[front.gain > 50].clipped_any.min()
print(f"the first point at a gain above 50 clips {100 * first_high:.1f}% of stars")

**The curve is steep, then flat.** From 0.5% to 1% clipped, SNR goes from 0.67 to 0.82 of the
best. From 2% to 5%, it goes only from 0.89 to 0.97, while the clipped share more than doubles. Most of the SNR is
bought cheaply; the last few percent cost a lot of stars. Where to stop is a taste for the target -
a star field wants the left end, a faint nebula the right - and MISSION leaves it there.

**The gain is not a choice until the clipping is very high.** Gain 100 and 200 only reach the curve
once roughly a tenth of the stars clip, and only for the last 1-2% of SNR.

## 2. Where the night's four settings sit

The four settings actually shot, read off `pareto_curve.csv`.

In [ ]:
CELLS = {"A": (50, 30.0), "B": (50, 120.0), "C": (200, 30.0), "D": (200, 120.0)}
shot = pd.DataFrame([{"cell": c, "gain": g, "t": t,
                      **curve[(curve.gain == g) & (curve.t == t)]
                      [["snr_rel_B", "clipped_any", "on_curve"]].iloc[0].to_dict()}
                     for c, (g, t) in CELLS.items()])
print(shot.round(4).to_string(index=False))

for cell, rival_gain in (("C", 50), ("D", 50)):
    c = shot.set_index("cell").loc[cell]
    better = curve[(curve.gain == rival_gain) & (curve.clipped_any <= c.clipped_any)]
    r = better.loc[better.snr_rel_B.idxmax()]
    print(f"\ncell {cell} (gain {int(c.gain)}, {c.t:.0f} s): SNR {c.snr_rel_B:.3f} at "
          f"{100 * c.clipped_any:.2f}% clipped.\n  gain {rival_gain} at {r.t:.0f} s clips "
          f"{100 * r.clipped_any:.2f}% and reaches {r.snr_rel_B:.3f}")

A and B, the two gain 50 cells, are on the curve. C and D, the two gain 200 cells, are not: for
each, a longer gain 50 sub clips no more stars and gives more SNR. The night that was shot to test
the model already holds the answer.

## 3. Why dead time decides it, not read noise

The usual reasoning says high gain wins because its read noise is low. On this sensor that is
true: read noise in electrons falls by about 3.5x across the HCG threshold at 200 (`24`). So why
does gain 50 win?

Because the comparison that matters is at **equal clipping**, not equal sub length. Gain 200 turns
each electron into about six times more counts, so a star's core reaches the ceiling about six
times sooner. To clip the same stars, a gain 200 sub must be about six times shorter. And this
night, every sub - short or long - costs 31.5 s of download, save and dither settle.

The model's SNR splits into two factors that can be read apart (`model.efficiency`):

- **exposing**: `sqrt(t / (t + t_dead))` - the share of the night the shutter is open.
- **sky-limited**: `sqrt(A t / (A t + R^2))` - the share of the noise that is sky, not read noise.

Here they are for cell C and the gain 50 setting that clips the same share of stars.

In [ ]:
F_SKY_B = mc["F_sky"]["value"]["B"]
DARK = mc["dark_current_bound"]["value"]
a = 1.01 * F_SKY_B + DARK                  # 31's faint signal: F_obj = 1% of sky


def factors(gain, t):
    r = M.read_noise_e(gain, mc["read_noise_counts"]["value"], mc["system_gain"]["value"])
    return {"gain": gain, "t, s": t, "read noise, e-": r,
            "exposing": math.sqrt(t / (t + T_DEAD)),
            "sky-limited": math.sqrt(a * t / (a * t + r * r))}


c = shot.set_index("cell").loc["C"]
match = curve[(curve.gain == 50) & (curve.clipped_any <= c.clipped_any)]
t50 = float(match.loc[match.snr_rel_B.idxmax(), "t"])
f = pd.DataFrame([factors(200, 30.0), factors(50, t50)])
f["product"] = f.exposing * f["sky-limited"]
print(f.round(3).to_string(index=False))

**Gain 200 wins the sky-limited factor and loses the exposing factor by much more.** At 30 s it is
almost perfectly sky-limited - its read noise is about one electron - but its shutter is open less
than half the night. Gain 50 carries about five electrons of read noise, which costs it about 6%;
in exchange its subs are long enough that the shutter is open most of the night.

**So the answer belongs to `t_dead`, and `t_dead` is a choice.** If the overhead were zero, the
exposing factor would be 1 for both, and gain 200's better sky-limited factor would win. 31.5 s is
what *dithering after every frame* costs (D83); dithering less often shortens it. Somewhere between
31.5 s and zero, the answer flips back to gain 200. Where, is the next notebook's question, and it
needs no bench night.

## 4. What the curve was drawn under

Each of these is in `31`'s purpose; here is which way each one pushes.

In [ ]:
print(f"every setting on the curve needs at least {front.n_subs.min()} subs "
      f"(stacking loss measured only to 18)")
print(f"the curve starts at {100 * front.clipped_any.min():.2f}% clipped")
print(f"the curve's best points at 1-5% clipped run {front[(front.clipped_any >= 0.01) & (front.clipped_any <= 0.05)].t.min():.0f}"
      f"-{front[(front.clipped_any >= 0.01) & (front.clipped_any <= 0.05)].t.max():.0f} s")

- **No stacking loss is assumed everywhere.** Every point needs more subs than the stacking loss
  was ever measured at. But the assumption can only *flatter* gain 200: it needs the shorter subs,
  so the bigger stacks, and any real loss that grows with stack size hurts it more. A measured loss
  would widen gain 50's lead.
- **Nothing below 0.24% clipped.** A handful of stars clip even at gain 50 and 30 s, and they are
  counted clipped at every setting, so the curve cannot go lower. If losing even those matters, the
  curve has no answer - only a shorter sub than it covers.
- **The ceiling is the safe one.** A star counts as clipped at `linear_to_at_least`, a few percent
  below 4095. Every clipped share here is slightly pessimistic, the same way at every setting.
- **Long subs, unproved mount.** The useful part of the curve is 100-500 s subs. Whether the mount
  tracks that long without trailing has never been measured. A sub that trails is worse than any
  point on this curve.
- **One field, one night.** Every star number belongs to NGC 7000, this night's seeing and this
  focus. A field with brighter stars moves the whole curve right; a softer night moves it left.

## 5. What is settled, and what is next

**Settled.**
- MISSION's deliverable exists for one target and one night, built only from constants this
  project measured, on a model whose two halves both passed their tests that same night.
- On this rig, with dithering after every frame, **gain 50 is the setting at every share of clipped
  stars below 9.5%**. The sub length is then the one real choice, and the curve prices it.
- `24`'s *"nothing recommends observing below 200"* does not survive real stars (D109).

**Not settled, named rather than fixed.**
1. **Where the gain answer flips with dead time.** Arithmetic on published constants; the cheapest
   question left, and the one that could change what is set in the ASIAIR.
2. **How long the mount tracks.** The curve's useful subs are long, and nothing has measured the
   limit.
3. **Other targets and nights.** The curve has to be drawn again from each one's own stars.